# Challenge 3: Workflow Orchestration

## From Manual Handoffs to Automated Pipelines

In Challenge 2, you manually passed outputs between agents. Now you'll use MAF's `WorkflowBuilder`
to create an **automated pipeline** with:

- **Sequential execution**: Triage → Diagnostics → Remediation → Verify → Comms
- **Conditional routing**: If verification fails → retry or escalate
- **Type-safe shared state**: A dataclass flows through the pipeline
- **Single entry point**: One call handles the entire incident

```
┌──────────┐    ┌──────────────┐    ┌─────────────┐    ┌──────────┐    ┌───────┐
│  Triage  │───►│ Diagnostics  │───►│ Remediation │───►│  Verify  │───►│ Comms │
└──────────┘    └──────────────┘    └─────────────┘    └────┬─────┘    └───────┘
                                           ▲                │
                                           │    FAIL        │
                                           └────────────────┘
```

---

## How This Challenge Works

1. **Shared State** is provided for you (the `IncidentState` dataclass)
2. **Triage executor** is provided as a reference
3. You build the **Diagnostics**, **Remediation**, and **Verification** executors
4. You implement the **conditional routing logic** (the hard part!)
5. You wire everything together with `WorkflowBuilder`

In [ ]:
import os
import sys
import json
from dataclasses import dataclass, field
sys.path.insert(0, "..")
from dotenv import load_dotenv

from agent_framework import WorkflowBuilder, WorkflowContext, executor
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human,
    get_health_status, run_smoke_test,
    post_to_slack, create_incident_ticket, update_status_page,
)

load_dotenv("../.env")

with open("../data/incidents.json") as f:
    incidents = json.load(f)

print("\u2705 Imports ready")

## Step 1: Shared State (Provided)

This dataclass holds everything as it flows through the pipeline.
Each executor reads from state and writes its results back.

In [ ]:
@dataclass
class IncidentState:
    """Shared state that flows through the incident response workflow."""
    # Input (set when workflow starts)
    alert_title: str = ""
    alert_service: str = ""
    alert_severity: str = ""
    alert_description: str = ""
    
    # Results from each stage (filled by executors)
    triage_result: str = ""
    diagnostics_result: str = ""
    remediation_result: str = ""
    verification_result: str = ""
    comms_result: str = ""
    
    # Control flow
    is_resolved: bool = False
    retry_count: int = 0
    max_retries: int = 1

print("\u2705 IncidentState defined")
print("   Fields:", [f.name for f in IncidentState.__dataclass_fields__.values()])

## Step 2: Triage Executor (REFERENCE)

Study this executor. Key things to notice:
1. `@executor` decorator marks it as a workflow step
2. `ctx: WorkflowContext[IncidentState]` gives access to shared state
3. It reads from state (`ctx.state.alert_title`)
4. It writes to state (`state.triage_result = result.text`)
5. It returns the **name of the next step** as a string

### Expected Output
```
============================================================
📋 TRIAGE STAGE
============================================================
[Agent calls check_alert_history, get_runbook]
Triage summary: Critical, recurring, auto-fix allowed...
→ Next: diagnostics
```

In [ ]:
@executor
async def triage_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """REFERENCE: First step — triage the incoming alert."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"\U0001f4cb TRIAGE STAGE")
    print(f"{'='*60}")
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="TriageAgent",
            instructions="""You are an incident Triage Agent. When given an alert:
1. Call check_alert_history with the service name
2. Call get_runbook with the incident type
Output: severity, recurring status, auto-remediation allowed, next steps.
Be concise and factual.""",
            tools=[check_alert_history, get_runbook],
        )
        
        result = await agent.run(
            f"Alert: {state.alert_title}\nService: {state.alert_service}\n"
            f"Severity: {state.alert_severity}\nDescription: {state.alert_description}"
        )
        
        state.triage_result = result.text
        print(f"\n{result.text[:500]}")
    
    return "diagnostics"  # Next step name

## Step 3: Diagnostics Executor (YOUR TURN)

Build the diagnostics executor. It should:
1. Read triage results from `ctx.state.triage_result`
2. Create a diagnostics agent with `get_metrics`, `get_logs`, `check_dependencies`
3. Pass both triage context and alert info in the prompt
4. Store the result in `state.diagnostics_result`
5. Return `"remediation"` as the next step

### Expected Output
```
============================================================
🔍 DIAGNOSTICS STAGE
============================================================
Root cause: pod-3 OOM, memory at 3891MB/4096MB, 4 restarts...
→ Next: remediation
```

In [ ]:
@executor
async def diagnostics_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """YOUR CODE: Diagnose the root cause."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"\U0001f50d DIAGNOSTICS STAGE")
    print(f"{'='*60}")
    
    # ╔══════════════════════════════════════════════════════════════╗
    # ║  YOUR CODE HERE                                             ║
    # ║  1. Create AzureCliCredential + AzureAIAgentClient          ║
    # ║  2. Create agent with tools: get_metrics, get_logs,         ║
    # ║     check_dependencies                                      ║
    # ║  3. Run agent with triage context + alert service           ║
    # ║  4. Store result in state.diagnostics_result                ║
    # ╚══════════════════════════════════════════════════════════════╝
    
    # TODO: Your implementation here
    pass
    
    return "remediation"

## Step 4: Remediation Executor (YOUR TURN)

Build the remediation executor. It should:
1. Read diagnostics results from `ctx.state.diagnostics_result`
2. Create agent with `restart_pod`, `scale_service`, `flush_cache`, `toggle_feature_flag`, `escalate_to_human`
3. Take action based on the root cause
4. Store result in `state.remediation_result`
5. Return `"verification"` as the next step

### Expected Output
```
============================================================
🔧 REMEDIATION STAGE (attempt 1)
============================================================
Action: Restarted payment-api-pod-3
Result: Success — pod restarted, new instance healthy
→ Next: verification
```

In [ ]:
@executor
async def remediation_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """YOUR CODE: Execute the fix."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"\U0001f527 REMEDIATION STAGE (attempt {state.retry_count + 1})")
    print(f"{'='*60}")
    
    # ╔══════════════════════════════════════════════════════════════╗
    # ║  YOUR CODE HERE                                             ║
    # ║  Same pattern as triage_executor but with fix tools         ║
    # ║  Store result in state.remediation_result                   ║
    # ╚══════════════════════════════════════════════════════════════╝
    
    # TODO: Your implementation here
    pass
    
    return "verification"

## Step 5: Verification Executor with ROUTING LOGIC (YOUR TURN — HARDEST PART)

This is the critical executor. Unlike the others, it has **conditional routing**:

```
if fix worked (RESOLVED):
    → go to "communications"     # Success path
elif retries remaining:
    → go to "remediation"         # Retry the fix
else:
    → go to "communications"      # Give up, notify team
```

### Requirements
1. Create verification agent with `get_health_status` + `run_smoke_test`
2. Tell the agent to end its response with `VERDICT: RESOLVED` or `VERDICT: FAILED`
3. Parse the verdict from the response
4. Update `state.is_resolved` and `state.retry_count`
5. Return the correct next step based on the routing logic above

### Expected Output (success path)
```
============================================================
✅ VERIFICATION STAGE
============================================================
Health: All pods healthy, latency 180ms
Smoke tests: 5/5 passing
VERDICT: RESOLVED
→ Next: communications (success)
```

In [ ]:
@executor
async def verification_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """YOUR CODE: Verify the fix AND implement routing logic."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"\u2705 VERIFICATION STAGE")
    print(f"{'='*60}")
    
    # ╔══════════════════════════════════════════════════════════════╗
    # ║  YOUR CODE HERE                                             ║
    # ║  1. Create verification agent (health + smoke test tools)   ║
    # ║  2. Tell it to end with VERDICT: RESOLVED or VERDICT: FAILED║
    # ║  3. Store result in state.verification_result               ║
    # ║  4. Parse the verdict and implement routing:                ║
    # ║     - RESOLVED → set state.is_resolved=True, return        ║
    # ║       "communications"                                      ║
    # ║     - FAILED + retries left → increment retry_count,       ║
    # ║       return "remediation"                                  ║
    # ║     - FAILED + no retries → return "communications"         ║
    # ╚══════════════════════════════════════════════════════════════╝
    
    # TODO: Your implementation here
    pass
    
    return "communications"  # Replace with your routing logic

## Step 6: Communications Executor (Provided)

This is the final step — it always returns `None` to end the workflow.

In [ ]:
@executor
async def communications_executor(ctx: WorkflowContext[IncidentState]) -> None:
    """Final step: Notify the team. Returns None to end workflow."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"\U0001f4e2 COMMUNICATIONS STAGE")
    print(f"{'='*60}")
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="CommunicationsAgent",
            instructions="""You are the Communications Agent. Notify the team.
1. Post to Slack (#incidents) — brief 4-line summary
2. Create an incident ticket with the full timeline
3. Update status page to 'operational' if resolved, 'degraded' if not""",
            tools=[post_to_slack, create_incident_ticket, update_status_page],
        )
        
        status = "RESOLVED" if state.is_resolved else "ESCALATED (auto-fix failed)"
        result = await agent.run(
            f"Status: {status}\n"
            f"Service: {state.alert_service}\n"
            f"Triage: {state.triage_result[:200]}\n"
            f"Diagnostics: {state.diagnostics_result[:200]}\n"
            f"Remediation: {state.remediation_result[:200]}\n"
            f"Verification: {state.verification_result[:200]}"
        )
        
        state.comms_result = result.text
        print(f"\n{result.text[:500]}")
    
    return None  # End of workflow

## Step 7: Wire It Together (YOUR TURN)

Use `WorkflowBuilder` to register all executors and build the workflow.

```python
workflow = (
    WorkflowBuilder[IncidentState]()
    .add_executor(executor_function, name="step_name")
    # ... add more
    .build()
)
```

The `name` parameter must match what executors return (e.g., triage returns `"diagnostics"`,
so the diagnostics executor must be registered with `name="diagnostics"`).

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  YOUR CODE: Build the workflow                              ║
# ║  Register all 5 executors with the correct names:           ║
# ║  triage, diagnostics, remediation, verification, comms      ║
# ╚══════════════════════════════════════════════════════════════╝

workflow = (
    WorkflowBuilder[IncidentState]()
    # TODO: Add all 5 executors here
    .build()
)

print("\u2705 Workflow built!")

## Step 8: Run It!

### Expected Output
```
🚨 INCIDENT RESPONSE INITIATED
   Alert: High Latency on payment-api
   Service: payment-api

============================================================
📋 TRIAGE STAGE
============================================================
...
============================================================
🔍 DIAGNOSTICS STAGE
============================================================
...
============================================================
🔧 REMEDIATION STAGE (attempt 1)
============================================================
...
============================================================
✅ VERIFICATION STAGE
============================================================
...
============================================================
📢 COMMUNICATIONS STAGE
============================================================
...

============================================================
🏁 INCIDENT RESPONSE COMPLETE
============================================================
   Resolved: ✅ Yes
   Retries: 0
   Stages completed: 5
```

In [ ]:
# Run the workflow on incident #1 (payment-api)
alert = incidents[0]

print(f"\U0001f6a8 INCIDENT RESPONSE INITIATED")
print(f"   Alert: {alert['title']}")
print(f"   Service: {alert['service']}")
print(f"   Severity: {alert['severity']}")

initial_state = IncidentState(
    alert_title=alert["title"],
    alert_service=alert["service"],
    alert_severity=alert["severity"],
    alert_description=alert["description"],
)

final_state = await workflow.run(state=initial_state)

print(f"\n\n{'='*60}")
print(f"\U0001f3c1 INCIDENT RESPONSE COMPLETE")
print(f"{'='*60}")
print(f"   Resolved: {'\u2705 Yes' if final_state.is_resolved else '\u274c No (escalated)'}")
print(f"   Retries: {final_state.retry_count}")
print(f"   Stages completed: {sum(1 for r in [final_state.triage_result, final_state.diagnostics_result, final_state.remediation_result, final_state.verification_result, final_state.comms_result] if r)}")

In [ ]:
# ✅ VALIDATION
checks = {
    "Triage completed": bool(final_state.triage_result),
    "Diagnostics completed": bool(final_state.diagnostics_result),
    "Remediation completed": bool(final_state.remediation_result),
    "Verification completed": bool(final_state.verification_result),
    "Communications completed": bool(final_state.comms_result),
    "Incident resolved": final_state.is_resolved,
}

print("\n\U0001f9ea WORKFLOW VALIDATION:")
for check, passed in checks.items():
    print(f"{'\u2705' if passed else '\u274c'} {check}")

if all(checks.values()):
    print("\n\U0001f389 Workflow executed perfectly!")
else:
    print("\n\u26a0\ufe0f Some stages didn't complete. Check your executor implementations.")

## Step 9: Prove It's Adaptive — Run Incident #3

The SAME workflow should make DIFFERENT decisions on a different incident.
Incident #3 is a notification-service email failure — the fix is toggling a feature flag, not restarting a pod.

### Expected Difference
| | Incident #1 (payment-api) | Incident #3 (notification-service) |
|---|---|---|
| Root cause | OOM on pod-3 | Email provider rate limiting |
| Remediation | `restart_pod` | `toggle_feature_flag` |
| Target | payment-api-pod-3 | use_backup_email → true |

In [ ]:
# Run on incident #3
alert3 = incidents[2]

print(f"\U0001f6a8 INCIDENT #3: {alert3['title']}")
print(f"   Service: {alert3['service']} | Severity: {alert3['severity']}")
print(f"   Description: {alert3['description']}")

state3 = IncidentState(
    alert_title=alert3["title"],
    alert_service=alert3["service"],
    alert_severity=alert3["severity"],
    alert_description=alert3["description"],
)

final_state3 = await workflow.run(state=state3)

print(f"\n\n{'='*60}")
print(f"\U0001f3c1 INCIDENT #3 COMPLETE")
print(f"{'='*60}")
print(f"   Resolved: {'\u2705' if final_state3.is_resolved else '\u274c'}")

# Check different decisions were made
print(f"\n\U0001f9ea DIFFERENT DECISIONS?")
remed_lower = final_state3.remediation_result.lower()
checks = {
    "Used toggle/feature_flag (not restart)": any(w in remed_lower for w in ["toggle", "feature", "flag", "backup"]),
    "Different root cause than incident #1": "oom" not in final_state3.diagnostics_result.lower() or "rate" in final_state3.diagnostics_result.lower(),
}
for check, passed in checks.items():
    print(f"{'\u2705' if passed else '\u274c'} {check}")

---
## What You Achieved

| Feature | Challenge 2 (Manual) | Challenge 3 (Workflow) |
|---------|---------------------|------------------------|
| Agent coordination | Manual copy-paste | Automated pipeline |
| Error handling | None | Retry + escalation |
| Routing | Fixed sequence | Conditional (verify → retry or complete) |
| State management | Ad-hoc variables | Typed `IncidentState` dataclass |
| Reusability | One-off scripts | Same workflow handles ANY incident |
| Entry point | 5 separate function calls | One `workflow.run()` call |

---

## \u27a1\ufe0f Next: Challenge 4 — Memory Patterns

The workflow works but starts from scratch every time. A human SRE remembers past incidents.
In Challenge 4, you'll add memory so the system gets **faster** with experience.

# 🔄 Step 3: Workflow Orchestration

## From Manual Handoffs to Automated Pipelines

In Step 2, we manually passed outputs between agents. Now we'll use MAF's `WorkflowBuilder`
to create an **automated incident response pipeline** with:

- **Sequential execution**: Triage → Diagnostics → Remediation → Verify → Comms
- **Conditional routing**: If verification fails → retry or escalate
- **Type-safe state**: Shared context flows through the pipeline
- **Single entry point**: One call handles the entire incident

---

## Architecture

```
┌──────────┐    ┌──────────────┐    ┌─────────────┐    ┌──────────┐    ┌───────┐
│  Triage  │───►│ Diagnostics  │───►│ Remediation │───►│  Verify  │───►│ Comms │
└──────────┘    └──────────────┘    └─────────────┘    └────┬─────┘    └───────┘
                                           ▲                │
                                           │    FAIL        │
                                           └────────────────┘
```

In [ ]:
import os
import sys
import json
from dataclasses import dataclass, field
sys.path.insert(0, "..")
from dotenv import load_dotenv

from agent_framework import WorkflowBuilder, WorkflowContext, executor
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human,
    get_health_status, run_smoke_test,
    post_to_slack, create_incident_ticket, update_status_page,
)

load_dotenv("../.env")

# Load incident data
with open("../data/incidents.json") as f:
    incidents = json.load(f)

print("✅ Imports ready")

## Define Shared State

The workflow needs shared state to pass information between executors.
We use a `dataclass` to hold the incident context as it flows through the pipeline.

In [ ]:
@dataclass
class IncidentState:
    """Shared state that flows through the incident response workflow."""
    # Input
    alert_title: str = ""
    alert_service: str = ""
    alert_severity: str = ""
    alert_description: str = ""
    
    # Accumulated context from each stage
    triage_result: str = ""
    diagnostics_result: str = ""
    remediation_result: str = ""
    verification_result: str = ""
    comms_result: str = ""
    
    # Control flow
    is_resolved: bool = False
    retry_count: int = 0
    max_retries: int = 1

print("✅ IncidentState defined")

## Define Workflow Executors

Each executor wraps one of our specialized agents. The `@executor` decorator
tells MAF this is a step in the workflow.

### 🎯 YOUR TASK
Complete the `triage_executor` and `diagnostics_executor` below.
The remediation, verification, and comms executors are provided as reference.

In [ ]:
@executor
async def triage_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """First step: Triage the incoming alert."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"📋 TRIAGE STAGE")
    print(f"{'='*60}")
    
    # TODO: Create the triage agent with check_alert_history and get_runbook tools
    # TODO: Run it with the alert details from state
    # TODO: Store result in state.triage_result
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="TriageAgent",
            instructions="""You are an incident Triage Agent. When given an alert:
1. Use check_alert_history to check if this is a recurring issue
2. Use get_runbook to find the remediation playbook
3. Output: severity, recurring status, whether auto-remediation is allowed, recommended incident type for runbook lookup.
Be concise.""",
            tools=[check_alert_history, get_runbook],
        )
        
        result = await agent.run(
            f"Alert: {state.alert_title}\nService: {state.alert_service}\n"
            f"Severity: {state.alert_severity}\nDescription: {state.alert_description}"
        )
        
        state.triage_result = result.text
        print(f"\n{result.text}")
    
    return "diagnostics"  # Next step

In [ ]:
@executor
async def diagnostics_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """Second step: Diagnose root cause."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"🔍 DIAGNOSTICS STAGE")
    print(f"{'='*60}")
    
    # TODO: Create the diagnostics agent with get_metrics, get_logs, check_dependencies
    # TODO: Include triage context in the prompt
    # TODO: Store result in state.diagnostics_result
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="DiagnosticsAgent",
            instructions="""You are a Diagnostics Agent. Investigate the incident using your tools.
1. Use get_metrics to check resource utilization
2. Use get_logs to find error patterns
3. Use check_dependencies to identify cascading failures
Output: specific root cause with evidence, affected components, and exact remediation action needed.""",
            tools=[get_metrics, get_logs, check_dependencies],
        )
        
        result = await agent.run(
            f"Triage summary:\n{state.triage_result}\n\n"
            f"Service: {state.alert_service}\nInvestigate root cause."
        )
        
        state.diagnostics_result = result.text
        print(f"\n{result.text}")
    
    return "remediation"  # Next step

In [ ]:
@executor
async def remediation_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """Third step: Execute the fix."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"🔧 REMEDIATION STAGE (attempt {state.retry_count + 1})")
    print(f"{'='*60}")
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="RemediationAgent",
            instructions="""You are a Remediation Agent. Execute the fix based on diagnostics.
Use the appropriate tool based on the root cause:
- OOM/memory leak → restart_pod
- High CPU/traffic → scale_service
- Stale cache → flush_cache
- External dependency failure → toggle_feature_flag for failover
- Cannot auto-fix → escalate_to_human
Use exact names from the diagnostics (specific pod names, service names, flag names).""",
            tools=[restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human],
        )
        
        result = await agent.run(
            f"Diagnostics findings:\n{state.diagnostics_result}\n\n"
            f"Service: {state.alert_service}\nExecute the appropriate fix."
        )
        
        state.remediation_result = result.text
        print(f"\n{result.text}")
    
    return "verification"  # Next step

In [ ]:
@executor
async def verification_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """Fourth step: Verify the fix worked. Routes to comms or back to remediation."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"✅ VERIFICATION STAGE")
    print(f"{'='*60}")
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="VerificationAgent",
            instructions="""You are a Verification Agent. Check if the fix worked.
1. Use get_health_status to check service health
2. Use run_smoke_test to run functional tests
End your response with exactly one of: VERDICT: RESOLVED or VERDICT: FAILED""",
            tools=[get_health_status, run_smoke_test],
        )
        
        result = await agent.run(
            f"Remediation performed:\n{state.remediation_result}\n\n"
            f"Service: {state.alert_service}\nVerify the fix worked."
        )
        
        state.verification_result = result.text
        print(f"\n{result.text}")
        
        # ROUTING LOGIC: Check if resolved or needs retry
        if "VERDICT: RESOLVED" in result.text.upper() or "RESOLVED" in result.text.upper():
            state.is_resolved = True
            return "communications"  # Success path
        else:
            state.retry_count += 1
            if state.retry_count >= state.max_retries:
                print("\n⚠️ Max retries reached — escalating to human")
                return "communications"  # Give up, notify team
            return "remediation"  # Retry the fix

In [ ]:
@executor
async def communications_executor(ctx: WorkflowContext[IncidentState]) -> None:
    """Final step: Notify team and create records."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"📢 COMMUNICATIONS STAGE")
    print(f"{'='*60}")
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="CommunicationsAgent",
            instructions="""You are the Communications Agent. Notify the team about the incident resolution.
1. Post a concise summary to Slack (#incidents channel) with severity based on resolution status
2. Create an incident ticket with the full timeline
3. Update the status page to 'operational' if resolved, 'degraded' if not
Keep the Slack message brief (4-5 lines max).""",
            tools=[post_to_slack, create_incident_ticket, update_status_page],
        )
        
        status = "RESOLVED" if state.is_resolved else "ESCALATED (auto-fix failed)"
        
        result = await agent.run(
            f"Incident status: {status}\n\n"
            f"Timeline:\n- Triage: {state.triage_result[:200]}\n"
            f"- Diagnostics: {state.diagnostics_result[:200]}\n"
            f"- Remediation: {state.remediation_result[:200]}\n"
            f"- Verification: {state.verification_result[:200]}\n\n"
            f"Service: {state.alert_service}\n"
            f"Notify the team."
        )
        
        state.comms_result = result.text
        print(f"\n{result.text}")
    
    return None  # End of workflow

## Build and Run the Workflow

### 🎯 YOUR TASK
Wire the executors together using `WorkflowBuilder`. The flow is:

```
triage → diagnostics → remediation → verification → communications
                                 ▲         │
                                 └─ FAIL ──┘
```

In [ ]:
# Build the workflow
workflow = (
    WorkflowBuilder[IncidentState]()
    .add_executor(triage_executor, name="triage")
    .add_executor(diagnostics_executor, name="diagnostics")
    .add_executor(remediation_executor, name="remediation")
    .add_executor(verification_executor, name="verification")
    .add_executor(communications_executor, name="communications")
    .build()
)

print("✅ Workflow built!")
print("   Flow: triage → diagnostics → remediation → verification → communications")
print("   Retry: verification can loop back to remediation if fix fails")

In [ ]:
# Run the workflow on incident #1 (payment-api high latency)
alert = incidents[0]

print(f"🚨 INCIDENT RESPONSE INITIATED")
print(f"   Alert: {alert['title']}")
print(f"   Service: {alert['service']}")
print(f"   Severity: {alert['severity']}")
print(f"\n{'='*60}")

# Initialize state with the alert data
initial_state = IncidentState(
    alert_title=alert["title"],
    alert_service=alert["service"],
    alert_severity=alert["severity"],
    alert_description=alert["description"],
)

# Run the full pipeline
final_state = await workflow.run(state=initial_state)

print(f"\n\n{'='*60}")
print(f"🏁 INCIDENT RESPONSE COMPLETE")
print(f"{'='*60}")
print(f"   Resolved: {'✅ Yes' if final_state.is_resolved else '❌ No (escalated)'}")
print(f"   Retries: {final_state.retry_count}")

## 🧪 Try a Different Incident

Run the same workflow on incident #3 (notification-service email failures).
Notice how the agents make **different decisions** — toggle feature flag instead of restart.

In [ ]:
# Try incident #3: notification-service email failures
alert2 = incidents[2]

print(f"🚨 INCIDENT RESPONSE INITIATED")
print(f"   Alert: {alert2['title']}")
print(f"   Service: {alert2['service']}")
print(f"   Severity: {alert2['severity']}")

state2 = IncidentState(
    alert_title=alert2["title"],
    alert_service=alert2["service"],
    alert_severity=alert2["severity"],
    alert_description=alert2["description"],
)

final_state2 = await workflow.run(state=state2)

print(f"\n\n🏁 COMPLETE — Resolved: {'✅' if final_state2.is_resolved else '❌'}")

## 🎉 What We Achieved

With workflow orchestration, we now have:

| Feature | Before (Step 2) | After (Step 3) |
|---------|----------------|----------------|
| Agent coordination | Manual copy-paste | Automated pipeline |
| Error handling | None | Retry + escalation |
| Routing | Fixed sequence | Conditional (verify → retry or complete) |
| State management | Ad-hoc variables | Typed `IncidentState` dataclass |
| Reusability | One-off scripts | Same workflow handles any incident |

---

## ➡️ Next: Step 4 — Memory Patterns

The workflow works, but it doesn't **learn**. Every incident starts from scratch.
In Step 4, we'll add memory so the system:
- Remembers past incidents and their resolutions
- Gets faster at diagnosing recurring issues
- Builds institutional knowledge over time